# 11 – Ajuste Fino de Modelo

Este notebook realiza ajuste fino de modelos de regresión para predecir `produccion_kg`.

## Objetivos

- cargar el dataset enriquecido
- usar un conjunto curado de variables (`selected_features`)
- separar train y test respetando el tiempo
- aplicar preprocesamiento robusto
- comparar modelos base y modelos afinados
- seleccionar el mejor modelo final


## 1. Librerías


In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)


## 2. Carga del dataset con features


In [ ]:
DATA_PATH = "../data/processed/training_dataset_features.parquet"

df = pd.read_parquet(DATA_PATH)

print("Shape:", df.shape)
display(df.head())


## 3. Definición de variable objetivo y selección curada de features

Aquí se usa una lista explícita de `selected_features` para mantener consistencia con el notebook de selección de modelo y evitar que se cuelen variables no deseadas o potencialmente problemáticas.


In [ ]:
target = "produccion_kg"

selected_features = [

    # producción histórica
    "produccion_roll_mean_3",
    "produccion_roll_std_3",

    # cambios
    "delta_produccion_1d",
    "delta_produccion_3d",

    # alimentación
    "consumo",
    "consumo_roll_mean_3",
    "sobrante",
    "sobrante_roll_mean_3",
    "rechazo",
    "rechazo_roll_mean_3",
    "kg_am",
    "kg_pm",
    "kg_totales",

    # ratios
    "ratio_consumo_oferta",
    "ratio_sobrante_consumo",
    "ratio_sobrante_oferta",

    # dieta
    "n_ingredientes",
    "usa_oro_balance",
    "usa_silo_avena",
    "usa_triticale",

    # clima
    "pressure_msl",

    # tiempo
    "mes_sin",
    "mes_cos",
    "dia_semana_sin",
    "dia_semana_cos",
    "dia_del_anio",

    # categórica
    "ubre"
]

# Solo dejar columnas que existan en el dataset
selected_features = [c for c in selected_features if c in df.columns]

print("Número de selected_features:", len(selected_features))
display(selected_features)

### Comentarios y observaciones

Si en esta etapa notas columnas casi vacías o muy sospechosas, se pueden excluir antes del ajuste fino para ahorrar tiempo de búsqueda.


## 4. Split temporal train / test

Se ordena por fecha, se conservan únicamente `selected_features + target`, se eliminan filas con NaN y luego se realiza un split temporal 80/20.


In [ ]:
df["date"] = pd.to_datetime(df["date"])

# Orden temporal
df = df.sort_values("date").copy()

# Mantener solo features + target y quitar NaN
data = df[selected_features + [target]].dropna().copy()

# Split temporal 80/20
split_idx = int(len(data) * 0.8)

train = data.iloc[:split_idx].copy()
test = data.iloc[split_idx:].copy()

print("Train:", train.shape)
print("Test:", test.shape)

## 5. Construcción de matrices X y y

A partir de `selected_features`, no de una lista automática de `features`.


In [ ]:
X_train = train[selected_features].copy()
y_train = train[target].copy()

X_test = test[selected_features].copy()
y_test = test[target].copy()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

### Nota

Este notebook asume que variables como `produccion_roll_mean_3`, `produccion_roll_std_3`, `delta_produccion_1d` y `delta_produccion_3d` ya fueron generadas correctamente en el dataset, idealmente usando solo información pasada (`shift`) para evitar fuga de información temporal.


## 6. Detección de columnas numéricas y categóricas


In [ ]:
num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numéricas:", len(num_features))
print("Categóricas:", len(cat_features))
display(cat_features)


## 7. Preprocesador robusto


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features),
    ],
    remainder="drop"
)

preprocessor


## 8. Métricas auxiliares


In [ ]:
def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


## 9. Validación temporal interna

Para ajuste fino se usa `TimeSeriesSplit` dentro del conjunto de entrenamiento.


In [ ]:
tscv = TimeSeriesSplit(n_splits=4)

print(tscv)


## 10. Modelos base para comparar antes del fine tuning


In [ ]:
baseline_models = {
    "RandomForest_base": RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ),
    "ExtraTrees_base": ExtraTreesRegressor(
        n_estimators=300,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ),
    "GradientBoosting_base": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        random_state=42
    )
}


## 11. Evaluación de modelos base


In [ ]:
baseline_results = {}
baseline_pipelines = {}

for name, model in baseline_models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    metrics = evaluate_regression(y_test, pred)
    baseline_results[name] = metrics
    baseline_pipelines[name] = pipe

baseline_results_df = (
    pd.DataFrame(baseline_results)
    .T
    .reset_index()
    .rename(columns={"index": "Modelo"})
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(baseline_results_df)


## 12. Espacios de búsqueda de hiperparámetros

Aquí se definen grids relativamente pequeños para mantener el tiempo de ejecución bajo control.


In [ ]:
search_spaces = {
    "RandomForest": {
        "model": RandomForestRegressor(random_state=42, n_jobs=-1),
        "param_grid": {
            "model__n_estimators": [200, 400],
            "model__max_depth": [8, 12, 16, None],
            "model__min_samples_split": [2, 5],
            "model__min_samples_leaf": [1, 2],
            "model__max_features": ["sqrt", 0.5]
        }
    },
    "ExtraTrees": {
        "model": ExtraTreesRegressor(random_state=42, n_jobs=-1),
        "param_grid": {
            "model__n_estimators": [200, 400],
            "model__max_depth": [8, 12, 16, None],
            "model__min_samples_split": [2, 5],
            "model__min_samples_leaf": [1, 2],
            "model__max_features": ["sqrt", 0.5]
        }
    },
    "GradientBoosting": {
        "model": GradientBoostingRegressor(random_state=42),
        "param_grid": {
            "model__n_estimators": [150, 300],
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__max_depth": [2, 3, 4],
            "model__subsample": [0.8, 1.0]
        }
    }
}


### Comentarios y observaciones

Si el entrenamiento tarda demasiado, puedes reducir:

- número de combinaciones
- número de splits
- número de estimadores


## 13. Búsqueda de hiperparámetros con GridSearchCV


In [ ]:
tuning_results = []
best_estimators = {}
grid_objects = {}

for name, config in search_spaces.items():
    print(f"\n=== Ajustando {name} ===")

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", config["model"])
    ])

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=config["param_grid"],
        scoring="neg_root_mean_squared_error",
        cv=tscv,
        n_jobs=-1,
        verbose=1,
        refit=True
    )

    grid.fit(X_train, y_train)

    best_pipe = grid.best_estimator_
    pred = best_pipe.predict(X_test)

    metrics = evaluate_regression(y_test, pred)

    row = {
        "Modelo": name,
        "best_cv_rmse": -grid.best_score_,
        "test_MAE": metrics["MAE"],
        "test_RMSE": metrics["RMSE"],
        "test_R2": metrics["R2"],
        "best_params": grid.best_params_
    }

    tuning_results.append(row)
    best_estimators[name] = best_pipe
    grid_objects[name] = grid


## 14. Resultados del fine tuning


In [ ]:
tuning_results_df = pd.DataFrame(tuning_results).sort_values("test_RMSE").reset_index(drop=True)
display(tuning_results_df)


## 15. Comparación entre modelos base y afinados


In [ ]:
baseline_comp = baseline_results_df.copy()
baseline_comp["tipo"] = "base"
baseline_comp = baseline_comp.rename(columns={"RMSE": "test_RMSE", "MAE": "test_MAE", "R2": "test_R2"})

tuned_comp = tuning_results_df[["Modelo", "test_MAE", "test_RMSE", "test_R2"]].copy()
tuned_comp["tipo"] = "ajustado"

comparison_df = pd.concat([baseline_comp[["Modelo", "test_MAE", "test_RMSE", "test_R2", "tipo"]], tuned_comp], ignore_index=True)
display(comparison_df.sort_values(["Modelo", "tipo"]))


## 16. Visualización comparativa


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=tuning_results_df, x="test_RMSE", y="Modelo")
plt.title("RMSE en test después de ajuste fino")
plt.xlabel("RMSE")
plt.ylabel("Modelo")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(data=tuning_results_df, x="test_R2", y="Modelo")
plt.title("R² en test después de ajuste fino")
plt.xlabel("R²")
plt.ylabel("Modelo")
plt.tight_layout()
plt.show()


## 17. Selección del mejor modelo final


In [ ]:
best_row = tuning_results_df.iloc[0]
best_model_name = best_row["Modelo"]
best_model = best_estimators[best_model_name]

print("Mejor modelo afinado:")
display(best_row)


## 18. Predicción real vs predicha del mejor modelo


In [ ]:
best_pred = best_model.predict(X_test)

comparison_plot_df = pd.DataFrame({
    "real": y_test.values,
    "predicho": best_pred
})

plt.figure(figsize=(6, 6))
sns.scatterplot(data=comparison_plot_df, x="real", y="predicho", alpha=0.4)
plt.title(f"Real vs predicho – {best_model_name}")
plt.xlabel("Producción real")
plt.ylabel("Producción predicha")

min_val = min(comparison_plot_df["real"].min(), comparison_plot_df["predicho"].min())
max_val = max(comparison_plot_df["real"].max(), comparison_plot_df["predicho"].max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

plt.tight_layout()
plt.show()


## 19. Residuales del mejor modelo


In [ ]:
residuals = y_test.values - best_pred

plt.figure(figsize=(8, 4))
sns.histplot(residuals, bins=40, kde=True)
plt.title(f"Distribución de residuales – {best_model_name}")
plt.xlabel("Residual")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.scatterplot(x=best_pred, y=residuals, alpha=0.4)
plt.axhline(0, linestyle="--")
plt.title(f"Predicción vs residual – {best_model_name}")
plt.xlabel("Predicción")
plt.ylabel("Residual")
plt.tight_layout()
plt.show()


## 20. Importancia de variables del mejor modelo (si aplica)


In [ ]:
model_step = best_model.named_steps["model"]

if hasattr(model_step, "feature_importances_"):
    feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
    importances = model_step.feature_importances_

    fi = (
        pd.DataFrame({
            "feature": feature_names,
            "importance": importances
        })
        .sort_values("importance", ascending=False)
        .head(25)
    )

    display(fi)

    plt.figure(figsize=(10, 8))
    sns.barplot(data=fi, x="importance", y="feature")
    plt.title(f"Top 25 variables más importantes – {best_model_name}")
    plt.xlabel("Importancia")
    plt.ylabel("Variable")
    plt.tight_layout()
    plt.show()
else:
    print(f"El modelo {best_model_name} no expone feature_importances_.")


## 21. Resumen final

Se presenta un resumen compacto del modelo final seleccionado.


In [ ]:
print("Modelo final seleccionado:", best_model_name)
print("Parámetros óptimos:")
print(grid_objects[best_model_name].best_params_)

final_metrics = evaluate_regression(y_test, best_pred)
print("\nMétricas finales en test:")
for k, v in final_metrics.items():
    print(f"{k}: {v:.4f}")


## 22. Conclusiones

En este notebook se realizó ajuste fino de hiperparámetros sobre varios modelos basados en árboles.

### Logros principales

- comparación de modelos base
- búsqueda de hiperparámetros con validación temporal
- evaluación sobre conjunto de test
- selección del mejor modelo final
- análisis visual de ajuste y residuales
- importancia de variables si el modelo lo permite

### Próximos pasos sugeridos

1. reducir o refinar el espacio de búsqueda para el mejor modelo  
2. comparar explícitamente con y sin variables de alimentación  
3. evaluar desempeño por vaca  
4. guardar el pipeline final entrenado  
5. crear un notebook de interpretación del modelo final
